# Reinforcement Learning 소개 · 실습


## 학습 목표

<!-- exercise-guidance:reinforcement-learning/week-01/objective -->
보상을 잘 받는 정책이 실제로 원하는 운전인지, 비용과 대기를 분리해 확인합니다.
<!-- /exercise-guidance:reinforcement-learning/week-01/objective -->


## 실행 조건

<!-- exercise-guidance:reinforcement-learning/week-01/execution -->
기준 Queue 예제는 NumPy로 실행합니다. 학습에 사용하지 않은 seed=100…104에서 정책을 비교합니다.
<!-- /exercise-guidance:reinforcement-learning/week-01/execution -->


## 적용 사례

<!-- exercise-guidance:reinforcement-learning/week-01/case-studies -->
### 장비·통신 자원의 처리량과 비용

합성 대기열에 매 시점 평균 2건이 도착합니다. 서비스 용량을 높이면 대기는 줄지만 에너지 비용이 늘어납니다.

**Observation·State:** 대기 수 q=0…20. 관측 q/20.

**행동:** 처리 용량 0·2·4 중 선택.

**Reward·Constraint:** −(다음 대기 수 + 0.2×용량² + 10×유실 수). 100 step에서 시간 제한.

**비교 기준:** 항상 용량 2, 대기 수에 따른 Threshold policy.

**확인할 지표:** 평가 Return, 평균 대기 수, 평균 용량², 유실 수.

### 에너지 저장장치 운전 설계

실측 자료를 연결하기 전, 가격·부하·저장량의 역할을 구분하는 연구 설계 사례입니다.

**Observation·State:** SOC, 부하, 가격, 시간. 미래 정보를 포함하지 않는 관측.

**행동:** 충·방전 전력. 이산 단계 또는 연속 범위를 먼저 결정.

**Reward·Constraint:** 전력 비용과 열화 비용. SOC·전력 한계의 처리 규칙 명시.

**비교 기준:** 시간대 규칙, 가능하면 동일 정보의 최적화.

**확인할 지표:** 총 비용, 제약 위반, 최대 수전 전력. 날짜 단위 Train/Test 분리.
<!-- /exercise-guidance:reinforcement-learning/week-01/case-studies -->


## 보상을 잘 받는 행동이 좋은 운전인가

<!-- exercise-guidance:reinforcement-learning/week-01/activity-1 -->
### 시작 조건

Queue의 두 정책을 기준으로 비용 계수 0.2의 의미를 살펴봅니다.

### 진행 방법

1. 대기·용량²·유실의 평가 평균을 분리해 표로 기록하세요.
2. 기록한 원시 지표에서 비용 계수 0.05·0.2·1.0일 때 Return을 다시 계산하세요.
3. 목적이 바뀌어 순위가 달라지는지 확인하고, 실제 연구에서 계수를 정할 근거를 적으세요.

### 남길 결과

- 두 정책의 비용–대기 산점도와 보상 계수 비교 표.
<!-- /exercise-guidance:reinforcement-learning/week-01/activity-1 -->


In [ ]:
"""Synthetic service queue. An action chooses capacity, including its energy cost."""
import numpy as np

class Queue:
    def reset(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.q, self.t = 0, 0
        return self.q

    def step(self, action):
        capacity = (0, 2, 4)[int(action)]
        arriving = int(self.rng.poisson(2))
        raw = max(0, self.q - capacity) + arriving
        dropped = max(0, raw - 20)
        self.q = min(20, raw)
        self.t += 1
        reward = -(self.q + .2 * capacity**2 + 10 * dropped)
        return self.q, reward, False, self.t >= 100, {'queue': self.q, 'capacity': capacity, 'dropped': dropped}

def evaluate(policy, seeds=range(100, 105)):
    rows = []
    for seed in seeds:
        env = Queue(); q = env.reset(seed)
        total = waiting = energy = dropped = 0
        while True:
            q, r, terminated, truncated, info = env.step(policy(q))
            total += r; waiting += q; energy += info['capacity']**2; dropped += info['dropped']
            if terminated or truncated: break
        rows.append([total, waiting / env.t, energy / env.t, dropped])
    return np.array(rows)

policies = {'constant': lambda q: 1, 'threshold': lambda q: 0 if q == 0 else (1 if q < 5 else 2)}
for name, policy in policies.items():
    print(name, evaluate(policy).mean(axis=0))


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-01/activity-1-record -->
**정책:**

**비용 계수:**

**평균 대기:**

**유실:**

**재계산 Return:**

### 확인할 점

- 같은 원시 지표를 이용한 재평가와 새 보상으로 재학습한 결과를 구분했는가?
- 유실을 낮게 평가해 발생할 실패를 설명할 수 있는가?
<!-- /exercise-guidance:reinforcement-learning/week-01/activity-1-record -->


## State가 부족한 상황

<!-- exercise-guidance:reinforcement-learning/week-01/activity-2 -->
### 시작 조건

도착률이 일정한 Queue에 이어, 도착률이 시간대별로 달라지는 상황을 설계합니다.

### 진행 방법

1. 같은 대기 수 q=4에서도 현재 도착률이 0.5 또는 4인 두 상황을 적으세요.
2. 관측이 q뿐인 경우와 q·시간대를 포함한 경우의 의사결정 차이를 설명하세요.
3. 부하·가격 등 실제로 사용 가능한 정보와 미래에만 알 수 있는 정보를 구분하세요.

### 남길 결과

- 상황 비교 2행과 관측 설계 표.
<!-- /exercise-guidance:reinforcement-learning/week-01/activity-2 -->


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-01/activity-2-record -->
**상황:**

**현재 관측:**

**빠진 정보:**

**예상 행동·이유:**

### 확인할 점

- 상태와 관측을 혼용하지 않았는가?
- 관측 확장이 실제로 얻을 수 있는 정보인가?
<!-- /exercise-guidance:reinforcement-learning/week-01/activity-2-record -->


## 본인의 연구 문제로 옮기기

<!-- exercise-guidance:reinforcement-learning/week-01/activity-3 -->
### 시작 조건

앞의 실험을 바탕으로 Queue·Motor 또는 본인의 연구 문제 하나를 선택해, 실제 적용을 위한 실험을 설계합니다.

### 진행 방법

1. State·Observation·Action·Reward·종료·시간 제한을 표 한 장으로 정의하세요. 각 변수의 단위와 실제로 얻을 수 있는 정보를 적습니다.
2. 예상 Transition 3개를 수치로 적어, 좋은 행동과 나쁜 행동의 보상이 의도에 맞는지 확인하세요.
3. 규칙 기반 비교 기준 한 개, 평가 조건 한 개, 가장 우려되는 실패 사례 한 개를 정하세요. 필요한 실측 자료나 Simulator도 적습니다.

### 남길 결과

- MDP 설계 표, Transition 예 3개, 비교 실험 계획.
- 실제 연구 자료가 없으면 적용 사례의 합성 조건으로 설계합니다.
<!-- /exercise-guidance:reinforcement-learning/week-01/activity-3 -->


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-01/activity-3-record -->
**항목:**

**구체적 정의·단위:**

**관측 가능 여부·확인 방법:**

### 확인할 점

- 미래 정보나 평가 Label을 관측에 넣지 않았는가?
- 학습 Return만으로 연구 기여를 판단하지 않고, 해당 도메인의 지표를 정했는가?
<!-- /exercise-guidance:reinforcement-learning/week-01/activity-3-record -->
